In [1]:
from pathlib import Path
import datetime as dt
import duckdb
import polars as pl

ROOT = Path(r"D:\Driveguard")
DATA = (ROOT / "data" / "processed").as_posix() + "/*.parquet"
MODEL = "ST12000NM0008"
HORIZON = 30   # predict: will it fail within the next 30 days?
SMART_ERR = ["smart_5_raw", "smart_187_raw", "smart_188_raw", "smart_197_raw", "smart_198_raw"]

df = duckdb.sql(f"""
SELECT * FROM read_parquet('{DATA}')
WHERE model = '{MODEL}'
ORDER BY serial_number, date
""").pl()
df.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(6780030, 12)

In [2]:
# When did each failed drive die?
fail_dates = (df.filter(pl.col("failure") == 1)
                .group_by("serial_number")
                .agg(pl.col("date").min().alias("fail_date")))

df = df.join(fail_dates, on="serial_number", how="left")
df = df.with_columns(
    (pl.col("fail_date") - pl.col("date")).dt.total_days().alias("days_to_failure")
).with_columns(
    pl.col("days_to_failure").is_between(0, HORIZON - 1).fill_null(False)
      .cast(pl.Int8).alias("label")
)

# Last 29 days: we can't know if healthy drives fail in July 2026, so drop those uncertain rows
cutoff = df["date"].max() - dt.timedelta(days=HORIZON - 1)
df = df.filter((pl.col("label") == 1) | (pl.col("date") < cutoff))

print(df.shape)
print("Positive rows (fail within 30 days):", df["label"].sum())

(6231127, 15)
Positive rows (fail within 30 days): 12747


In [3]:
df = df.with_columns(
    [pl.col(c).cast(pl.Float64).fill_null(0) for c in SMART_ERR] +
    [pl.col(c).cast(pl.Float64) for c in ["smart_9_raw", "smart_194_raw"]]
)

# FIX: Seagate packs SMART 188 as three 16-bit counters; keep only the timeout count
df = df.with_columns(
    (pl.col("smart_188_raw").cast(pl.Int64) & 0xFFFF).cast(pl.Float64).alias("smart_188_raw")
)

feats = []
for c in SMART_ERR:
    n = c.replace("_raw", "")
    feats += [
        (pl.col(c) - pl.col(c).rolling_min_by("date", window_size="8d").over("serial_number")).alias(f"{n}_diff7"),
        (pl.col(c) - pl.col(c).rolling_min_by("date", window_size="31d").over("serial_number")).alias(f"{n}_diff30"),
        pl.col(c).rolling_max_by("date", window_size="30d").over("serial_number").alias(f"{n}_max30"),
        (pl.col(c) > 0).cast(pl.Int32).rolling_sum_by("date", window_size="30d")
            .over("serial_number").alias(f"{n}_days_nonzero30"),
    ]

feats += [
    (pl.col("smart_9_raw") / 24).alias("age_days"),
    pl.col("smart_194_raw").alias("temp_c"),
    pl.col("smart_194_raw").rolling_mean_by("date", window_size="7d")
        .over("serial_number").alias("temp_mean7"),
    pl.col("date").diff().over("serial_number").dt.total_days().fill_null(1).alias("gap_days"),
]

df = df.with_columns(feats)
df.shape

(6231127, 39)

In [4]:
out = ROOT / "data" / "features_ST12000NM0008.parquet"
df.drop("fail_date", "failure").write_parquet(out)
print("Saved ->", out)

Saved -> D:\Driveguard\data\features_ST12000NM0008.parquet


In [5]:
one = df.filter(pl.col("label") == 1)["serial_number"][0]
(df.filter(pl.col("serial_number") == one)
   .select("date", "label", "days_to_failure", "smart_5_raw", "smart_5_diff7",
           "smart_197_raw", "smart_5_days_nonzero30")
   .tail(10))

date,label,days_to_failure,smart_5_raw,smart_5_diff7,smart_197_raw,smart_5_days_nonzero30
date,i8,i64,f64,f64,f64,i32
2025-08-11,1,25,6088.0,1016.0,184.0,30
2025-08-12,1,24,6152.0,1032.0,168.0,30
2025-08-13,1,23,6168.0,1016.0,176.0,30
2025-08-14,1,22,6168.0,760.0,176.0,30
2025-08-15,1,21,6168.0,728.0,176.0,30
2025-08-16,1,20,6168.0,592.0,176.0,30
2025-08-17,1,19,6168.0,208.0,176.0,30
2025-08-18,1,18,6168.0,80.0,176.0,30
2025-08-23,1,13,6176.0,8.0,176.0,26
